---

## 📦 Imports et Configuration

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import pickle
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Deepchecks NLP
from deepchecks.nlp import TextData
from deepchecks.nlp.suites import model_evaluation

# ML
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.model_selection import train_test_split
import joblib

In [3]:
# Configuration des chemins
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'testing' else Path.cwd()
PROCESSOR_DIR = BASE_DIR / 'processors'
MODELS_DIR = BASE_DIR / 'models'
TESTING_DIR = BASE_DIR / 'testing'
TESTING_DIR.mkdir(parents=True, exist_ok=True)

print("="*80)
print("🏆 DEEPCHECKS NLP - NIVEAU 3 : PERFORMANCE DU MODÈLE")
print("="*80)
print(f"📅 Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"📁 Base: {BASE_DIR}")
print()

🏆 DEEPCHECKS NLP - NIVEAU 3 : PERFORMANCE DU MODÈLE
📅 Date: 2025-12-15 01:46:12
📁 Base: e:\MLOps\mlops_election



---

## 📥 Chargement des Données et du Modèle

In [4]:
def load_preprocessed_data():
    """Charge les données preprocessées"""
    print("📦 Chargement des données")
    print("-" * 80)
    
    data_path = PROCESSOR_DIR / 'preprocessed_data.pkl'
    if not data_path.exists():
        raise FileNotFoundError(
            f"Données non trouvées: {data_path}\n"
            "Exécutez: python scripts/preprocess.py"
        )
    
    with open(data_path, 'rb') as f:
        data = pickle.load(f)
    
    print(f"✅ Données chargées:")
    print(f"   Train: {data['X_train'].shape}")
    print(f"   Val:   {data['X_val'].shape}")
    print(f"   Test:  {data['X_test'].shape}")
    print()
    
    return data

In [5]:
def load_cleaned_texts():
    """Charge les textes nettoyés"""
    texts_path = PROCESSOR_DIR / 'cleaned_texts.pkl'
    if not texts_path.exists():
        raise FileNotFoundError(f"Textes non trouvés: {texts_path}")
    
    with open(texts_path, 'rb') as f:
        data = pickle.load(f)
    
    print(f"✅ Textes chargés: {len(data['cleaned'])} textes")
    return data['cleaned'], data['labels']

In [6]:
def load_best_model():
    """Charge le meilleur modèle ML"""
    print("🤖 Chargement du meilleur modèle")
    print("-" * 80)
    
    # Essayer de trouver le meilleur modèle
    model_files = list(MODELS_DIR.glob('*.pkl'))
    
    if not model_files:
        print("⚠️  Aucun modèle trouvé - chargement du LogisticRegression par défaut")
        model_path = MODELS_DIR / 'model_lr.pkl'
    else:
        # Pour l'exemple, on prend Logistic Regression
        model_path = MODELS_DIR / 'model_lr.pkl'
        if not model_path.exists():
            model_path = model_files[0]
    
    if model_path.exists():
        model = joblib.load(model_path)
        print(f"✅ Modèle chargé: {model_path.name}")
        print(f"   Type: {type(model).__name__}")
        print()
        return model
    else:
        raise FileNotFoundError("Aucun modèle disponible")

In [7]:
# Charger les données
data = load_preprocessed_data()
texts, labels = load_cleaned_texts()
model = load_best_model()

📦 Chargement des données
--------------------------------------------------------------------------------
✅ Données chargées:
   Train: (2403, 5000)
   Val:   (515, 5000)
   Test:  (516, 5000)

✅ Textes chargés: 3434 textes
🤖 Chargement du meilleur modèle
--------------------------------------------------------------------------------
✅ Modèle chargé: model_gradient_boosting.pkl
   Type: GradientBoostingClassifier



---

## 📝 Création des TextData pour Deepchecks NLP

In [8]:
def create_text_data(texts_list, labels_list, split_name='train'):
    """Crée un TextData Deepchecks NLP à partir de textes et labels"""
    print(f"📝 Création TextData NLP ({split_name})")
    print("-" * 80)
    
    text_data = TextData(
        raw_text=texts_list,
        label=labels_list,
        task_type='text_classification',
        name=f'{split_name}_dataset'
    )
    
    print(f"✅ TextData créé:")
    print(f"   Nombre de textes: {len(texts_list)}")
    print(f"   Distribution labels: {pd.Series(labels_list).value_counts().to_dict()}")
    print()
    
    return text_data

In [9]:
# Créer le split train/test (même split que preprocess.py)
df = pd.DataFrame({'texts': texts, 'labels': labels})
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df['labels']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['labels']
)

print(f"Split effectué:")
print(f"  Train: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
print(f"  Val:   {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")
print(f"  Test:  {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")
print()

Split effectué:
  Train: 2403 (70.0%)
  Val:   515 (15.0%)
  Test:  516 (15.0%)



In [10]:
# Créer les TextData NLP
train_text_data = create_text_data(
    train_df['texts'].tolist(), 
    train_df['labels'].tolist(), 
    'train'
)
test_text_data = create_text_data(
    test_df['texts'].tolist(), 
    test_df['labels'].tolist(), 
    'test'
)

📝 Création TextData NLP (train)
--------------------------------------------------------------------------------
✅ TextData créé:
   Nombre de textes: 2403
   Distribution labels: {0: 1233, 1: 1170}

📝 Création TextData NLP (test)
--------------------------------------------------------------------------------
✅ TextData créé:
   Nombre de textes: 516
   Distribution labels: {0: 265, 1: 251}



---

## 🏆 NIVEAU 3 : MODEL PERFORMANCE NLP

### Suite d'évaluation de modèle Deepchecks NLP

La suite `model_evaluation()` exécute automatiquement les checks suivants :
1. **Prediction Drift** : Compare les distributions de prédictions train vs test
2. **Train Test Performance** : Compare les métriques train vs test (overfitting)
3. **Property Segments Performance** : Performance par segments de données
4. **Metadata Segments Performance** : Performance par métadonnées (si disponibles)

In [11]:
def run_nlp_model_performance(model, train_data, test_data, X_train, X_test):
    """NIVEAU 3: Évaluation de la performance du modèle NLP"""
    print("\n" + "="*80)
    print("📊 NIVEAU 3: MODEL PERFORMANCE NLP")
    print("="*80)
    
    # Faire les prédictions
    print("\n🔮 Génération des prédictions...")
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Calculer les probabilités si possible
    try:
        y_train_proba = model.predict_proba(X_train)
        y_test_proba = model.predict_proba(X_test)
        has_proba = True
    except:
        y_train_proba = None
        y_test_proba = None
        has_proba = False

    # Normaliser y_true et y_pred en numpy arrays
    y_train_true = np.array(train_data.label)
    y_test_true = np.array(test_data.label)

    # Aligner les types de prédictions avec les labels
    def _align_preds_to_labels(preds, labels):
        preds_arr = np.array(preds).ravel()
        if len(labels) == 0:
            return preds_arr
        sample = labels[0]
        # Si labels sont des strings, convertir preds en str
        if isinstance(sample, str):
            return np.array([str(p) for p in preds_arr])
        # Si labels sont des ints, convertir preds en int
        if isinstance(sample, (int, np.integer)):
            try:
                return np.array([int(p) for p in preds_arr])
            except Exception:
                return preds_arr
        return preds_arr

    y_train_pred = _align_preds_to_labels(y_train_pred, y_train_true)
    y_test_pred = _align_preds_to_labels(y_test_pred, y_test_true)

    # Créer TextData pour les checks de performance
    train_data_with_pred = TextData(
        raw_text=train_data.text,
        label=train_data.label,
        task_type='text_classification',
        name='train_with_predictions'
    )
    test_data_with_pred = TextData(
        raw_text=test_data.text,
        label=test_data.label,
        task_type='text_classification',
        name='test_with_predictions'
    )
    
    print("\n✅ Prédictions générées")
    print(f"   Train predictions shape: {y_train_pred.shape}")
    print(f"   Test predictions shape: {y_test_pred.shape}")
    
    # Calculer les propriétés pour les checks de performance
    print("\n⏳ Calcul des propriétés textuelles pour l'évaluation...")
    train_data_with_pred.calculate_builtin_properties()
    test_data_with_pred.calculate_builtin_properties()
    print("✅ Propriétés calculées")
    
    # Suite d'évaluation du modèle NLP
    performance_suite = model_evaluation()
    
    print("\n🔍 Checks de Performance (suite complète):")
    print("   1. Prediction Drift")
    print("   2. Train Test Performance")
    print("   3. Property Segments Performance")
    print("   4. Metrics sklearn (intégrés)")
    
    # Exécuter la suite avec prédictions passées en arguments
    print("\n⏳ Exécution de la suite d'évaluation...")
    try:
        # Passer les prédictions directement à la méthode run()
        performance_result = performance_suite.run(
            train_dataset=train_data_with_pred, 
            test_dataset=test_data_with_pred,
            train_predictions=list(y_train_pred),
            test_predictions=list(y_test_pred),
            train_probabilities=y_train_proba if has_proba else None,
            test_probabilities=y_test_proba if has_proba else None
        )
        print("✅ Suite d'évaluation exécutée")
    except Exception as e:
        print(f"   ⚠️  Suite d'évaluation erreur: {str(e)[:150]}")
        print("   ℹ️  Les métriques de performance sont calculées ci-dessous")
        performance_result = None
    
    # Métriques custom
    print("\n🏆 Métriques du Modèle:")
    
    # Déterminer pos_label selon le type de label
    sample_label = train_data.label[0]
    if isinstance(sample_label, str):
        pos_label = '1'
    else:
        pos_label = 1
    
    # Train metrics
    train_acc = accuracy_score(train_data.label, y_train_pred)
    train_f1 = f1_score(train_data.label, y_train_pred, average='binary', pos_label=pos_label)
    
    # Test metrics
    test_acc = accuracy_score(test_data.label, y_test_pred)
    test_f1 = f1_score(test_data.label, y_test_pred, average='binary', pos_label=pos_label)
    test_precision = precision_score(test_data.label, y_test_pred, average='binary', pos_label=pos_label)
    test_recall = recall_score(test_data.label, y_test_pred, average='binary', pos_label=pos_label)
    
    print(f"   Train Accuracy: {train_acc:.4f}")
    print(f"   Train F1:       {train_f1:.4f}")
    print(f"   Test Accuracy:  {test_acc:.4f}")
    print(f"   Test F1:        {test_f1:.4f}")
    print(f"   Test Precision: {test_precision:.4f}")
    print(f"   Test Recall:    {test_recall:.4f}")
    
    # Overfitting check
    overfit_gap = train_acc - test_acc
    print(f"\n⚠️  Écart Train/Test: {overfit_gap:.4f}")
    if overfit_gap > 0.1:
        print("   ⚠️  ATTENTION: Possible overfitting détecté!")
    else:
        print("   ✅ Pas d'overfitting majeur")
    
    # Confusion matrix
    cm = confusion_matrix(test_data.label, y_test_pred)
    print(f"\n📊 Matrice de Confusion (Test):")
    print(cm)
    
    # Classification report
    print(f"\n📋 Classification Report (Test):")
    print(classification_report(test_data.label, y_test_pred, 
                                target_names=['Classe 0', 'Classe 1'], 
                                digits=4))
    
    # Sauvegarder le rapport
    performance_report_path = TESTING_DIR / 'deepchecks_nlp_performance_report.html'
    if performance_result:
        performance_result.save_as_html(str(performance_report_path))
        print(f"\n✅ Rapport de performance NLP sauvegardé: {performance_report_path.name}")
    
    return performance_result, {
        'train_acc': train_acc,
        'test_acc': test_acc,
        'test_f1': test_f1,
        'test_precision': test_precision,
        'test_recall': test_recall,
        'overfit_gap': overfit_gap
    }

In [12]:
# Exécuter les checks de performance NLP
performance_result, metrics = run_nlp_model_performance(
    model, train_text_data, test_text_data,
    data['X_train'], data['X_test']
)


📊 NIVEAU 3: MODEL PERFORMANCE NLP

🔮 Génération des prédictions...

✅ Prédictions générées
   Train predictions shape: (2403,)
   Test predictions shape: (516,)

⏳ Calcul des propriétés textuelles pour l'évaluation...


100%|██████████| 33/33 [00:00<00:00, 91.47it/s]
deepchecks - WARNING - Could not find model's classes, using the observed classes. In order to make sure the classes used by the model are inferred correctly, please use the model_classes argument


✅ Propriétés calculées

🔍 Checks de Performance (suite complète):
   1. Prediction Drift
   2. Train Test Performance
   3. Property Segments Performance
   4. Metrics sklearn (intégrés)

⏳ Exécution de la suite d'évaluation...


✅ Suite d'évaluation exécutée

🏆 Métriques du Modèle:
   Train Accuracy: 0.8552
   Train F1:       0.8422
   Test Accuracy:  0.7364
   Test F1:        0.7056
   Test Precision: 0.7725
   Test Recall:    0.6494

⚠️  Écart Train/Test: 0.1187
   ⚠️  ATTENTION: Possible overfitting détecté!

📊 Matrice de Confusion (Test):
[[217  48]
 [ 88 163]]

📋 Classification Report (Test):
              precision    recall  f1-score   support

    Classe 0     0.7115    0.8189    0.7614       265
    Classe 1     0.7725    0.6494    0.7056       251

    accuracy                         0.7364       516
   macro avg     0.7420    0.7341    0.7335       516
weighted avg     0.7412    0.7364    0.7343       516


✅ Rapport de performance NLP sauvegardé: deepchecks_nlp_performance_report.html


In [13]:
# Afficher le widget interactif de performance
if performance_result:
    performance_result
else:
    print("⚠️  Pas de résultat Deepchecks disponible")
    print("Les métriques sklearn sont affichées ci-dessus")

---

## 📊 Analyse détaillée des métriques

In [14]:
print("\n" + "="*80)
print("📊 ANALYSE DÉTAILLÉE DES MÉTRIQUES")
print("="*80)

print("\n🎯 Performance Globale:")
print(f"   Accuracy (Test):  {metrics['test_acc']:.2%}")
print(f"   F1-Score (Test):  {metrics['test_f1']:.2%}")
print(f"   Precision (Test): {metrics['test_precision']:.2%}")
print(f"   Recall (Test):    {metrics['test_recall']:.2%}")

print("\n⚖️  Équilibre Precision/Recall:")
if metrics['test_precision'] > metrics['test_recall']:
    print("   Le modèle est conservateur (peu de faux positifs)")
elif metrics['test_recall'] > metrics['test_precision']:
    print("   Le modèle est permissif (peu de faux négatifs)")
else:
    print("   Le modèle est équilibré")

print("\n📈 Généralisation:")
print(f"   Train Accuracy: {metrics['train_acc']:.2%}")
print(f"   Test Accuracy:  {metrics['test_acc']:.2%}")
print(f"   Gap:            {metrics['overfit_gap']:.2%}")

if metrics['overfit_gap'] < 0.05:
    print("   ✅ Excellente généralisation")
elif metrics['overfit_gap'] < 0.10:
    print("   ✅ Bonne généralisation")
else:
    print("   ⚠️  Overfitting détecté")


📊 ANALYSE DÉTAILLÉE DES MÉTRIQUES

🎯 Performance Globale:
   Accuracy (Test):  73.64%
   F1-Score (Test):  70.56%
   Precision (Test): 77.25%
   Recall (Test):    64.94%

⚖️  Équilibre Precision/Recall:
   Le modèle est conservateur (peu de faux positifs)

📈 Généralisation:
   Train Accuracy: 85.52%
   Test Accuracy:  73.64%
   Gap:            11.87%
   ⚠️  Overfitting détecté


---

## 📊 Résumé et Recommandations

In [15]:
print("\n" + "="*80)
print("✅ NIVEAU 3 : PERFORMANCE DU MODÈLE - TERMINÉ")
print("="*80)

print("\n📂 Rapport généré:")
print(f"   {TESTING_DIR / 'deepchecks_nlp_performance_report.html'}")

print("\n💡 Recommandations:")

if metrics['overfit_gap'] > 0.1:
    print("\n   ⚠️  OVERFITTING DÉTECTÉ - Actions recommandées:")
    print("      • Augmenter les données d'entraînement")
    print("      • Appliquer une régularisation plus forte")
    print("      • Réduire la complexité du modèle")
    print("      • Utiliser data augmentation")

if metrics['test_f1'] < 0.7:
    print("\n   ⚠️  F1-SCORE FAIBLE - Actions recommandées:")
    print("      • Ajouter des features NLP (n-grams, embeddings)")
    print("      • Fine-tuner TunBERT sur le domaine")
    print("      • Améliorer le nettoyage des données")
    print("      • Équilibrer les classes")

if metrics['test_precision'] < 0.7:
    print("\n   ⚠️  PRECISION FAIBLE - Trop de faux positifs:")
    print("      • Ajuster le seuil de décision")
    print("      • Ajouter des features discriminantes")

if metrics['test_recall'] < 0.7:
    print("\n   ⚠️  RECALL FAIBLE - Trop de faux négatifs:")
    print("      • Équilibrer les classes (SMOTE)")
    print("      • Ajuster les poids de classes")

if metrics['overfit_gap'] <= 0.1 and metrics['test_f1'] >= 0.7:
    print("\n   ✅ MODÈLE PERFORMANT:")
    print("      • Généralisation excellente")
    print("      • Performance satisfaisante")
    print("      • Prêt pour la production")
    print("      • Considérer le déploiement")

print("\n🔗 Ouvrez le rapport HTML pour visualiser les détails")
print("="*80)


✅ NIVEAU 3 : PERFORMANCE DU MODÈLE - TERMINÉ

📂 Rapport généré:
   e:\MLOps\mlops_election\testing\deepchecks_nlp_performance_report.html

💡 Recommandations:

   ⚠️  OVERFITTING DÉTECTÉ - Actions recommandées:
      • Augmenter les données d'entraînement
      • Appliquer une régularisation plus forte
      • Réduire la complexité du modèle
      • Utiliser data augmentation

   ⚠️  RECALL FAIBLE - Trop de faux négatifs:
      • Équilibrer les classes (SMOTE)
      • Ajuster les poids de classes

🔗 Ouvrez le rapport HTML pour visualiser les détails


---

## 📚 Documentation

### Checks exécutés

| Check | Description | Critère |
|-------|-------------|---------|
| **Prediction Drift** | Distribution prédictions train vs test | Drift < threshold |
| **Train Test Performance** | Métriques train vs test | Gap < 10% |
| **Property Segments Performance** | Performance par segments | All > 70% acc |
| **Confusion Matrix** | Matrice de confusion | Balanced |

### Métriques calculées

- **Accuracy** : Précision globale
- **F1-Score** : Moyenne harmonique précision/rappel
- **Precision** : Taux de vrais positifs (1 - faux positifs)
- **Recall** : Taux de détection (1 - faux négatifs)
- **Overfitting Gap** : Écart train/test accuracy

### Interprétation

#### Overfitting
- **Gap < 5%** : Excellente généralisation ✅
- **Gap 5-10%** : Bonne généralisation ✅
- **Gap > 10%** : Overfitting ⚠️

#### F1-Score
- **F1 > 80%** : Excellent modèle ✅
- **F1 70-80%** : Bon modèle ✅
- **F1 < 70%** : Modèle à améliorer ⚠️

#### Precision vs Recall
- **Precision élevée** : Peu de faux positifs (conservateur)
- **Recall élevé** : Peu de faux négatifs (permissif)
- **Équilibrés** : Bon compromis ✅

### Validation complète

Pour une validation complète, exécutez les 3 niveaux :
1. ✅ **NIVEAU 1** : `deepchecks_integrity.ipynb` - Intégrité des données
2. ✅ **NIVEAU 2** : `deepchecks_distribution.ipynb` - Drift et distribution
3. ✅ **NIVEAU 3** : `deepchecks_performance.ipynb` - Performance du modèle

### Ressources
- [Deepchecks NLP Docs](https://docs.deepchecks.com/stable/nlp/index.html)
- [Model Evaluation Suite](https://docs.deepchecks.com/stable/nlp/auto_checks/model_evaluation/index.html)
- [DEEPCHECKS_NLP_DOCUMENTATION.md](DEEPCHECKS_NLP_DOCUMENTATION.md)